# 06 — Unified Shot Type Label Mapping

Both datasets label the same physical strokes differently:

| Dataset | JSON field | Language | # types |
|---|---|---|---|
| FineBadminton | `hit_type` | English | 11 |
| ShuttleSet | `type` | Traditional Chinese | 19+ |

This notebook validates the **17-class unified vocabulary** defined in `src/config.py` (based on the official ShuttleSet English translations) and confirms coverage across both datasets.

| idx | Unified label | ShuttleSet (Chinese) | FineBadminton |
|---|---|---|---|
| 0 | `short_serve` | 發短球 | serve |
| 1 | `long_serve` | 發長球 | — |
| 2 | `smash` | 殺球 | kill |
| 3 | `tap_smash` | 點扣 | net kill |
| 4 | `push_rush` | 推撲球, 撲球 | — |
| 5 | `clear` | 長球 | clear |
| 6 | `slice_drop` | 切球 | — |
| 7 | `net_drop` | 放小球 | drop shot |
| 8 | `transition` | 過渡球, 過度切球 | — |
| 9 | `drive` | 平球, 後場抽平球, 防守回抽, 小平球 | drive |
| 10 | `block` | 擋小球 | block |
| 11 | `lob_lift` | 挑球 | net lift |
| 12 | `defensive_lift` | 防守回挑 | — |
| 13 | `cross_net` | 勾球 | cross-court net shot |
| 14 | `net_shot` | 網前球 | net shot |
| 15 | `smash_defense` | 接殺防守 | — |
| 16 | `push` | 推球 | push shot |

In [1]:
import sys, json
from pathlib import Path
from collections import Counter
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))

from src.config import (
    FB_ANNOTATIONS, SS_OUTPUTS,
    UNIFIED_SHOT_TYPES, UNIFIED_SHOT_TO_IDX, NUM_UNIFIED_SHOT_TYPES,
    FB_HIT_TYPE_TO_UNIFIED,
    SS_SHOT_TYPE_TO_UNIFIED,
)

print(f"Unified classes ({NUM_UNIFIED_SHOT_TYPES}): {UNIFIED_SHOT_TYPES}")

Unified classes (9): ['serve', 'smash', 'clear', 'drop', 'drive', 'net_shot', 'lift', 'push', 'block']


## 1. FineBadminton — `hit_type` coverage

In [2]:
with open(FB_ANNOTATIONS) as f:
    fb_data = json.load(f)

fb_hits = []
for rally in fb_data:
    for hit in rally.get('hitting', []):
        fb_hits.append(hit.get('hit_type', ''))

fb_counts = Counter(fb_hits)

rows = []
for raw, count in fb_counts.most_common():
    unified = FB_HIT_TYPE_TO_UNIFIED.get(raw)
    rows.append({'hit_type (raw)': raw or '(empty)', 'count': count,
                 'unified': unified or '⚠ unmapped', 'idx': UNIFIED_SHOT_TO_IDX.get(unified, '—')})

fb_df = pd.DataFrame(rows)
print(f"Total FB shots: {len(fb_hits)}")
mapped = sum(1 for h in fb_hits if FB_HIT_TYPE_TO_UNIFIED.get(h))
print(f"Mapped to unified: {mapped}/{len(fb_hits)} ({mapped/len(fb_hits)*100:.1f}%)")
fb_df

Total FB shots: 414
Mapped to unified: 413/414 (99.8%)


,hit_type (raw),count,unified,idx
0,push shot,84,push,7
1,kill,70,smash,1
2,net shot,44,net_shot,5
3,block,43,block,8
4,serve,40,serve,0
5,drive,34,drive,4
6,clear,31,clear,2
7,drop shot,23,drop,3
8,cross-court net shot,20,net_shot,5
9,net lift,16,lift,6


## 2. ShuttleSet — `type` coverage

In [3]:
ss_types = Counter()
for json_file in sorted(SS_OUTPUTS.glob('*.json')):
    if json_file.name == 'pipeline_summary.json':
        continue
    with open(json_file) as f:
        records = json.load(f)
    for r in records:
        ss_types[r.get('type', '')] += 1

total_ss = sum(ss_types.values())
rows = []
for raw, count in ss_types.most_common():
    unified = SS_SHOT_TYPE_TO_UNIFIED.get(raw)
    rows.append({'type (Chinese)': raw or '(empty)', 'count': count,
                 'pct': f"{count/total_ss*100:.1f}%",
                 'unified': unified or '— (excluded)', 'idx': UNIFIED_SHOT_TO_IDX.get(unified, '—')})

ss_df = pd.DataFrame(rows)
mapped_ss = sum(c for t, c in ss_types.items() if SS_SHOT_TYPE_TO_UNIFIED.get(t))
print(f"Total SS shots: {total_ss}")
print(f"Mapped to unified: {mapped_ss}/{total_ss} ({mapped_ss/total_ss*100:.1f}%)")
ss_df

Total SS shots: 21191
Mapped to unified: 20576/21191 (97.1%)


,type (Chinese),count,pct,unified,idx
0,放小球,3823,18.0%,drop,3
1,挑球,3159,14.9%,lift,6
2,擋小球,2145,10.1%,block,8
3,推球,1686,8.0%,push,7
4,長球,1609,7.6%,clear,2
5,殺球,1405,6.6%,smash,1
6,發短球,1328,6.3%,serve,0
7,切球,1208,5.7%,drop,3
8,點扣,1013,4.8%,smash,1
9,勾球,842,4.0%,net_shot,5


## 3. Unified class distribution (both datasets combined)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Count per unified class from each dataset
fb_unified = Counter(FB_HIT_TYPE_TO_UNIFIED.get(h) for h in fb_hits if FB_HIT_TYPE_TO_UNIFIED.get(h))
ss_unified = Counter(SS_SHOT_TYPE_TO_UNIFIED.get(t) for t, c in ss_types.items()
                     if SS_SHOT_TYPE_TO_UNIFIED.get(t) for _ in range(c))

classes = UNIFIED_SHOT_TYPES
fb_vals = [fb_unified.get(c, 0) for c in classes]
ss_vals = [ss_unified.get(c, 0) for c in classes]

x = np.arange(len(classes))
w = 0.35
fig, ax = plt.subplots(figsize=(16, 5))
ax.bar(x - w/2, fb_vals, w, label='FineBadminton (hit_type)', color='#4f86c6')
ax.bar(x + w/2, ss_vals, w, label='ShuttleSet (type)', color='#f4845f')
ax.set_xticks(x)
ax.set_xticklabels(classes, rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Shot count')
ax.set_title('Unified shot type distribution (17-class) — FineBadminton vs ShuttleSet')
ax.legend()
plt.tight_layout()
plt.show()

print("\nUnified counts (FB / SS):")
for c, fv, sv in zip(classes, fb_vals, ss_vals):
    note = ''
    if fv == 0: note = '  ← FB has no equivalent'
    if sv == 0: note = '  ← SS has no equivalent'
    print(f"  [{UNIFIED_SHOT_TO_IDX[c]:>2}] {c:<18}: FB={fv:>4}  SS={sv:>5}{note}")

## 4. Verify dataset.py correctly resolves unified indices

In [ ]:
from src.config import SS_SKELETONS_GDINO
from src.data.dataset import FineBadmintonDataset, ShuttleSetDataset

# --- FineBadminton ---
fb_ds = FineBadmintonDataset()
fb_with_unified = sum(1 for info in fb_ds.rally_info if info['unified_shot_type_idx'] is not None)
print(f"FB: {fb_with_unified}/{len(fb_ds.rally_info)} shots have unified shot type labels")

# Show a sample
for info in fb_ds.rally_info[:3]:
    print(f"  hit_type='{info['hit_type']}' → unified='{info['unified_shot_type']}' (idx={info['unified_shot_type_idx']})")

In [ ]:
# --- ShuttleSet ---
ss_ds = ShuttleSetDataset(load_shot_types=True)

def _get_idx(s):
    if isinstance(s, dict): return s.get('shot_type_idx')
    return s[1] if isinstance(s, tuple) else None

ss_with_unified = sum(1 for s in ss_ds.samples if _get_idx(s) is not None)
total = len(ss_ds.samples)
print(f"SS: {ss_with_unified}/{total} shots have unified shot type labels ({ss_with_unified/total*100:.1f}%)")

# Distribution of unified indices in SS
idx_counts = Counter(_get_idx(s) for s in ss_ds.samples if _get_idx(s) is not None)
print("\nUnified class distribution in SS dataset:")
for idx in range(NUM_UNIFIED_SHOT_TYPES):
    print(f"  [{idx}] {UNIFIED_SHOT_TYPES[idx]:<15}: {idx_counts.get(idx, 0)}")